# US natural gas: weekly storage data

**Goal of the project:** does Henry Hub gas react to the EIA weekly storage report (Thursdays, 10:30 New York time), and does a bigger "surprise" mean a bigger price move?

**This notebook (01)** downloads and cleans the data, then saves it to `data/` so notebooks 02 and 03 can load it instantly:
- Weekly US (Lower 48) working gas in storage from the EIA API v2
- Daily Henry Hub (`NG=F`) and Brent (`BZ=F`) futures prices from Yahoo Finance
- A proxy for the storage "surprise": actual weekly change minus the 5-year average change for the same week

## 1. Setup
The EIA API key is read from `Projects/.env` (git-ignored), so it never appears in the code or on GitHub.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # searches upward from this folder to Projects/.env
EIA_API_KEY = os.environ.get("EIA_API_KEY")
assert EIA_API_KEY, "EIA_API_KEY not found - check Projects/.env"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
START = "2010-01-01"

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## 2. Weekly gas storage (EIA)
Each value is the total working gas in Lower 48 underground storage (Bcf) at the end of a week ending on **Friday**. EIA publishes it the following **Thursday**.

Storage is the market's balance sheet: summer injections refill it, winter withdrawals draw it down. When storage is below normal for the time of year, the market is tight and prices tend to be higher.

In [2]:
EIA_URL = "https://api.eia.gov/v2/natural-gas/stor/wkly/data/"
LOWER48_SERIES = "NW2_EPG0_SWO_R48_BCF"


def fetch_eia_storage(start=START):
    params = {
        "api_key": EIA_API_KEY,
        "frequency": "weekly",
        "data[0]": "value",
        "facets[series][]": LOWER48_SERIES,
        "start": start,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "length": 5000,
    }
    r = requests.get(EIA_URL, params=params, timeout=30)
    if not r.ok:
        # don't use raise_for_status(): its message includes the URL, i.e. the API key
        raise RuntimeError(f"EIA request failed with status {r.status_code}")
    df = pd.DataFrame(r.json()["response"]["data"])
    return (
        df.assign(period=pd.to_datetime(df["period"]), value=pd.to_numeric(df["value"]))
        .set_index("period")["value"]
        .rename("storage_bcf")
        .sort_index()
    )


storage = fetch_eia_storage()
storage.to_csv(DATA_DIR / "eia_storage_weekly.csv")
print(storage.tail(), f"\n{len(storage)} weeks")

period
2026-09-25    3415
2026-10-02    3500
2026-10-09    3567
2026-10-16    3644
2026-10-23    3724
Name: storage_bcf, dtype: int64 
878 weeks


## 3. The surprise proxy
Free consensus forecasts don't exist, so the "expected" change here is the **average change for the same week over the previous 5 years**. Only past years are used, so there's no look-ahead.

- **Negative surprise:** a smaller injection or bigger withdrawal than normal, so the market is tighter. **Bullish.**
- **Positive surprise:** a looser market. **Bearish.**

**Caveat:** traders forecast each week's number from weather, so much of this "surprise" is already priced in before the release. Expect a weak relationship. The interesting part is how weak, and on which days it does work.

In [3]:
df = storage.to_frame()
df["change_bcf"] = df["storage_bcf"].diff()

iso = df.index.isocalendar()  # ISO year/week, so late-Dec weeks don't land in the wrong year
df["year"], df["week"] = iso["year"].astype(int), iso["week"].astype(int)

by_week = df.pivot_table(index="year", columns="week", values="change_bcf")
avg5 = by_week.shift(1).rolling(5, min_periods=5).mean()  # prior 5 years only
df = df.join(avg5.stack().rename("change_5y_avg"), on=["year", "week"])

df["surprise_bcf"] = df["change_bcf"] - df["change_5y_avg"]
df["report_date"] = df.index + pd.Timedelta(days=6)  # Fri week-ending -> Thu release (holidays shift this)

df.to_csv(DATA_DIR / "storage_surprise.csv")
df.dropna().tail(8)

,storage_bcf,change_bcf,year,week,change_5y_avg,surprise_bcf,report_date
period,,,,,,,
2026-09-04,3254,40.0,2026,36,65.6,-25.6,2026-09-10
2026-09-11,3298,44.0,2026,37,78.2,-34.2,2026-09-17
2026-09-18,3351,53.0,2026,38,81.0,-28.0,2026-09-24
2026-09-25,3415,64.0,2026,39,88.0,-24.0,2026-10-01
2026-10-02,3500,85.0,2026,40,90.4,-5.4,2026-10-08
2026-10-09,3567,67.0,2026,41,91.4,-24.4,2026-10-15
2026-10-16,3644,77.0,2026,42,75.6,1.4,2026-10-22
2026-10-23,3724,80.0,2026,43,80.4,-0.4,2026-10-29


## 4. Daily prices
The report comes out at 10:30 New York time and NYMEX gas settles at 14:30, so **Thursday's close-to-close return already contains the reaction.** Friday's return tells us something else: whether the move continued or reversed.

Brent is included as a comparison: oil and gas are both energy, but they have separate supply and demand balances.

In [4]:
prices = yf.download(["NG=F", "BZ=F"], start=START, progress=False, auto_adjust=False)["Close"]
prices = prices.rename(columns={"NG=F": "henry_hub", "BZ=F": "brent"}).dropna(how="all")
prices.to_csv(DATA_DIR / "prices_daily.csv")

returns = np.log(prices / prices.shift(1))
returns["is_report_day"] = returns.index.isin(df["report_date"])
print(returns["is_report_day"].sum(), "report days matched to trading days out of", df["report_date"].ge(START).sum(), "weeks")
prices.tail()

/opt/anaconda3/lib/python3.12/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/opt/anaconda3/lib/python3.12/site-packages/yfinance/scrapers/history.py:147: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)


855 report days matched to trading days out of 878 weeks


Ticker,brent,henry_hub
Date,,
2026-10-26,115.809998,2.602
2026-10-27,113.879997,2.697
2026-10-28,115.320000,2.817
2026-10-29,114.879997,2.952
2026-10-30,113.430000,2.898
